# pLM4CPP-XAI — Manuscript Figure Generation

Run the core pipeline first. These cells regenerate the final manuscript-oriented
visualizations from the saved result tables. Because the original study was developed
iteratively in Colab, each figure cell should be run only after its required source
tables have been generated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/pLM4CPP_XAI_2026')
print('Project:', PROJECT)


In [ ]:
# ======================================================================
# MANUSCRIPT FIGURE 2 — FINAL IMPROVED VERSION
# Larger + bold X/Y tick values and clearer publication formatting
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score
)

# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

PRED_DIR = PROJECT / "05_predictions"

OUT_DIR = (
    PROJECT
    / "07_results"
    / "final_manuscript_package"
    / "03_main_figures"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# MODELS
# ======================================================================

MODEL_DIRS = {
    "ESM2-320": "ESM2_320",
    "ESM2-640": "ESM2_640",
    "ESM2-1280": "ESM2_1280",
    "ProtT5": "ProtT5",
}

MODEL_ORDER = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean ensemble",
    "Median ensemble"
]


# ======================================================================
# COLUMN DETECTION
# ======================================================================

def find_column(df, candidates):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    for c in df.columns:
        lc = c.lower()

        for candidate in candidates:
            if candidate.lower() in lc:
                return c

    return None


def detect_columns(df):

    label_col = find_column(
        df,
        [
            "label",
            "true_label",
            "y_true",
            "target",
            "class",
            "actual"
        ]
    )

    prob_col = find_column(
        df,
        [
            "probability",
            "prob",
            "pred_prob",
            "prediction_probability",
            "positive_probability",
            "cpp_probability",
            "score",
            "y_score"
        ]
    )

    pred_col = find_column(
        df,
        [
            "predicted_label",
            "prediction",
            "pred_label",
            "y_pred"
        ]
    )

    seq_col = find_column(
        df,
        [
            "sequence",
            "peptide",
            "seq"
        ]
    )

    id_col = find_column(
        df,
        [
            "sequence_id",
            "id",
            "peptide_id"
        ]
    )

    return {
        "label": label_col,
        "prob": prob_col,
        "pred": pred_col,
        "sequence": seq_col,
        "id": id_col
    }


# ======================================================================
# LOAD PREDICTION FILE
# ======================================================================

def load_prediction_file(path):

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

    df = pd.read_csv(path)

    cols = detect_columns(df)

    if cols["label"] is None:
        raise KeyError(
            f"\nCould not detect true-label column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    if cols["prob"] is None:
        raise KeyError(
            f"\nCould not detect probability column in:\n{path}\n\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    return df, cols


# ======================================================================
# LOAD ALL SAVED PREDICTIONS
# ======================================================================

datasets = {
    "validation": {},
    "internal": {},
    "external": {}
}

dataset_files = {
    "validation": "validation_predictions.csv",
    "internal": "internal_test_predictions.csv",
    "external": "kelm_external_predictions.csv"
}


print("=" * 90)
print("LOADING SAVED MODEL PREDICTIONS")
print("=" * 90)

for model_name, folder_name in MODEL_DIRS.items():

    print(f"\n{model_name}")

    for dataset_name, filename in dataset_files.items():

        path = (
            PRED_DIR
            / folder_name
            / filename
        )

        df, cols = load_prediction_file(path)

        datasets[dataset_name][model_name] = {
            "df": df,
            "cols": cols
        }

        print(
            f"  {dataset_name:10s}: "
            f"{len(df):4d} rows | "
            f"label={cols['label']} | "
            f"prob={cols['prob']} | "
            f"pred={cols['pred']}"
        )


# ======================================================================
# EXTRACT TRUE LABELS AND MODEL SCORES
# ======================================================================

def extract_dataset(dataset_name):

    scores = {}

    first_model = list(MODEL_DIRS.keys())[0]

    first = datasets[dataset_name][first_model]

    y_true = pd.to_numeric(
        first["df"][first["cols"]["label"]],
        errors="raise"
    ).astype(int).to_numpy()

    for model in MODEL_DIRS:

        entry = datasets[dataset_name][model]

        model_y = pd.to_numeric(
            entry["df"][entry["cols"]["label"]],
            errors="raise"
        ).astype(int).to_numpy()

        if not np.array_equal(y_true, model_y):

            raise ValueError(
                f"Label ordering mismatch detected for "
                f"{dataset_name}: {model}"
            )

        scores[model] = pd.to_numeric(
            entry["df"][entry["cols"]["prob"]],
            errors="raise"
        ).to_numpy()

    score_matrix = np.column_stack(
        [
            scores["ESM2-320"],
            scores["ESM2-640"],
            scores["ESM2-1280"],
            scores["ProtT5"]
        ]
    )

    scores["Mean ensemble"] = np.mean(
        score_matrix,
        axis=1
    )

    scores["Median ensemble"] = np.median(
        score_matrix,
        axis=1
    )

    return y_true, scores


validation_y_true, validation_scores = extract_dataset(
    "validation"
)

internal_y_true, internal_scores = extract_dataset(
    "internal"
)

external_y_true, external_scores = extract_dataset(
    "external"
)


print("\n✓ Scores reconstructed successfully")

print("\nDataset sizes:")
print("Validation      :", len(validation_y_true))
print("Internal test   :", len(internal_y_true))
print("KELM external   :", len(external_y_true))


# ======================================================================
# VALIDATION-OPTIMAL THRESHOLDS
# ======================================================================

def optimize_threshold_mcc(y_true, probabilities):

    thresholds_scan = np.linspace(
        0.01,
        0.99,
        981
    )

    best_threshold = 0.5
    best_mcc = -999

    for threshold in thresholds_scan:

        pred = (
            probabilities >= threshold
        ).astype(int)

        mcc = matthews_corrcoef(
            y_true,
            pred
        )

        if mcc > best_mcc:

            best_mcc = mcc
            best_threshold = threshold

    return best_threshold, best_mcc


thresholds = {}

print("\n" + "=" * 90)
print("VALIDATION-OPTIMAL THRESHOLDS")
print("=" * 90)

for model in MODEL_ORDER:

    threshold, validation_mcc = optimize_threshold_mcc(
        validation_y_true,
        validation_scores[model]
    )

    thresholds[model] = threshold

    print(
        f"{model:18s} "
        f"threshold = {threshold:.3f} | "
        f"validation MCC = {validation_mcc:.3f}"
    )


# ======================================================================
# CALCULATE METRICS
# ======================================================================

def calculate_metrics(
    y_true,
    scores,
    thresholds
):

    metrics = {}

    for model in MODEL_ORDER:

        prob = scores[model]

        threshold = thresholds[model]

        pred = (
            prob >= threshold
        ).astype(int)

        metrics[model] = {

            "roc_auc":
                roc_auc_score(
                    y_true,
                    prob
                ),

            "pr_auc":
                average_precision_score(
                    y_true,
                    prob
                ),

            "mcc":
                matthews_corrcoef(
                    y_true,
                    pred
                ),

            "bacc":
                balanced_accuracy_score(
                    y_true,
                    pred
                )
        }

    return metrics


internal_metrics = calculate_metrics(
    internal_y_true,
    internal_scores,
    thresholds
)

external_metrics = calculate_metrics(
    external_y_true,
    external_scores,
    thresholds
)


# ======================================================================
# PRINT METRICS
# ======================================================================

summary_rows = []

for model in MODEL_ORDER:

    summary_rows.append({

        "Model": model,

        "Internal ROC-AUC":
            internal_metrics[model]["roc_auc"],

        "Internal PR-AUC":
            internal_metrics[model]["pr_auc"],

        "Internal MCC":
            internal_metrics[model]["mcc"],

        "Internal bACC":
            internal_metrics[model]["bacc"],

        "KELM ROC-AUC":
            external_metrics[model]["roc_auc"],

        "KELM PR-AUC":
            external_metrics[model]["pr_auc"],

        "KELM MCC":
            external_metrics[model]["mcc"],

        "KELM bACC":
            external_metrics[model]["bacc"]
    })


summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("FIGURE 2 METRICS")
print("=" * 90)

display(
    summary_df.round(3)
)


# ======================================================================
# PUBLICATION STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 13,

    # Panel titles
    "axes.titlesize": 16,
    "axes.titleweight": "bold",

    # X/Y axis titles
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # X/Y tick values
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,

    # Tick thickness
    "xtick.major.width": 1.4,
    "ytick.major.width": 1.4,

    # Legend
    "legend.fontsize": 10,

    # Axes
    "axes.linewidth": 1.4,

    # Curves
    "lines.linewidth": 2.2,

    # Save
    "savefig.dpi": 600
})


# ======================================================================
# FIGURE LAYOUT
# ======================================================================

fig = plt.figure(
    figsize=(19.5, 12)
)

gs = fig.add_gridspec(
    2,
    3,
    width_ratios=[
        1.12,
        1.12,
        1.08
    ],
    wspace=0.30,
    hspace=0.37
)


axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axE = fig.add_subplot(gs[0, 2])

axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axF = fig.add_subplot(gs[1, 2])


# ======================================================================
# PANEL A — INTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["roc_auc"]
    )

    axA.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axA.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axA.set_xlim(0, 1)
axA.set_ylim(0, 1.02)

axA.set_xlabel(
    "False-positive rate"
)

axA.set_ylabel(
    "True-positive rate"
)

axA.set_title(
    "Internal test ROC",
    pad=12
)

leg = axA.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL B — INTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        internal_y_true,
        internal_scores[model]
    )

    auc_value = (
        internal_metrics[model]["pr_auc"]
    )

    axB.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


internal_prevalence = np.mean(
    internal_y_true
)

axB.axhline(
    internal_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({internal_prevalence:.3f})"
)

axB.set_xlim(0, 1)
axB.set_ylim(0, 1.02)

axB.set_xlabel(
    "Recall"
)

axB.set_ylabel(
    "Precision"
)

axB.set_title(
    "Internal test precision–recall",
    pad=12
)

leg = axB.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL C — KELM EXTERNAL ROC
# ======================================================================

for model in MODEL_ORDER:

    fpr, tpr, _ = roc_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["roc_auc"]
    )

    axC.plot(
        fpr,
        tpr,
        label=f"{model} ({auc_value:.3f})"
    )


axC.plot(
    [0, 1],
    [0, 1],
    ":",
    linewidth=1.4
)

axC.set_xlim(0, 1)
axC.set_ylim(0, 1.02)

axC.set_xlabel(
    "False-positive rate"
)

axC.set_ylabel(
    "True-positive rate"
)

axC.set_title(
    "KELM external ROC",
    pad=12
)

leg = axC.legend(
    title="Model (ROC-AUC)",
    loc="lower right",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# PANEL D — KELM EXTERNAL PRECISION–RECALL
# ======================================================================

for model in MODEL_ORDER:

    precision, recall, _ = precision_recall_curve(
        external_y_true,
        external_scores[model]
    )

    auc_value = (
        external_metrics[model]["pr_auc"]
    )

    axD.plot(
        recall,
        precision,
        label=f"{model} ({auc_value:.3f})"
    )


external_prevalence = np.mean(
    external_y_true
)

axD.axhline(
    external_prevalence,
    linestyle=":",
    linewidth=1.4,
    label=f"Prevalence ({external_prevalence:.3f})"
)

axD.set_xlim(0, 1)
axD.set_ylim(0, 1.02)

axD.set_xlabel(
    "Recall"
)

axD.set_ylabel(
    "Precision"
)

axD.set_title(
    "KELM external precision–recall",
    pad=12
)

leg = axD.legend(
    title="Model (PR-AUC)",
    loc="lower left",
    frameon=False,
    labelspacing=0.38,
    handlelength=2.5
)

leg.get_title().set_fontweight("bold")
leg.get_title().set_fontsize(10.5)


# ======================================================================
# BAR CHART SETTINGS
# ======================================================================

x = np.arange(
    len(MODEL_ORDER)
)

width = 0.34

bar_labels = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
    "Mean\nensemble",
    "Median\nensemble"
]


# ======================================================================
# PANEL E — MCC
# ======================================================================

int_mcc = [
    internal_metrics[m]["mcc"]
    for m in MODEL_ORDER
]

ext_mcc = [
    external_metrics[m]["mcc"]
    for m in MODEL_ORDER
]


bars_int = axE.bar(
    x - width/2,
    int_mcc,
    width,
    label="Internal test"
)

bars_ext = axE.bar(
    x + width/2,
    ext_mcc,
    width,
    label="KELM external"
)


axE.set_ylabel(
    "Matthews correlation coefficient"
)

axE.set_title(
    "Matthews correlation coefficient",
    pad=12
)

axE.set_xticks(x)

axE.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axE.set_ylim(
    0.50,
    max(
        max(int_mcc),
        max(ext_mcc)
    ) + 0.10
)

axE.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axE.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.008,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# PANEL F — BALANCED ACCURACY
# ======================================================================

int_bacc = [
    internal_metrics[m]["bacc"]
    for m in MODEL_ORDER
]

ext_bacc = [
    external_metrics[m]["bacc"]
    for m in MODEL_ORDER
]


bars_int = axF.bar(
    x - width/2,
    int_bacc,
    width,
    label="Internal test"
)

bars_ext = axF.bar(
    x + width/2,
    ext_bacc,
    width,
    label="KELM external"
)


axF.set_ylabel(
    "Balanced accuracy"
)

axF.set_title(
    "Balanced accuracy",
    pad=12
)

axF.set_xticks(x)

axF.set_xticklabels(
    bar_labels,
    rotation=27,
    ha="right"
)

axF.set_ylim(
    0.70,
    min(
        1.00,
        max(
            max(int_bacc),
            max(ext_bacc)
        ) + 0.08
    )
)

axF.legend(
    frameon=False,
    loc="upper right",
    fontsize=10.5
)


for bars in [
    bars_int,
    bars_ext
]:

    for bar in bars:

        value = bar.get_height()

        axF.text(
            bar.get_x()
            + bar.get_width()/2,

            value + 0.006,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold"
        )


# ======================================================================
# AXIS STYLE
# Larger + bold numeric X/Y tick values
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
    axE,
    axF
]:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.4)
    ax.spines["bottom"].set_linewidth(1.4)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=13,
        width=1.4,
        length=5
    )

    # Bold all X-axis tick labels
    for label in ax.get_xticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )

    # Bold all Y-axis tick labels
    for label in ax.get_yticklabels():

        label.set_fontweight(
            "bold"
        )

        label.set_fontsize(
            13
        )


# ======================================================================
# Slightly smaller X labels for E/F model names
# ======================================================================

for label in axE.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


for label in axF.get_xticklabels():

    label.set_fontsize(12)
    label.set_fontweight("bold")


# ======================================================================
# PANEL LABELS
# ======================================================================

panel_labels = [
    (axA, "A"),
    (axB, "B"),
    (axC, "C"),
    (axD, "D"),
    (axE, "E"),
    (axF, "F")
]


for ax, label in panel_labels:

    ax.text(
        -0.16,
        1.08,

        label,

        transform=ax.transAxes,

        fontsize=22,
        fontweight="bold",

        va="top",
        ha="left"
    )


# ======================================================================
# FINAL SPACING
# ======================================================================

fig.subplots_adjust(
    left=0.062,
    right=0.985,
    bottom=0.115,
    top=0.94,
    wspace=0.31,
    hspace=0.38
)


# ======================================================================
# SAVE FINAL FIGURE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_2_Model_performance_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

fig.savefig(
    svg_file,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FINAL FIGURE 2 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 3A — REPRESENTATIVE CPP XAI MAP
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelweight": "bold",
    "xtick.labelsize": 8,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.1,
    "savefig.dpi": 600
})


fig, axA = plt.subplots(
    figsize=(14.5, 3.4)
)


imA = axA.imshow(
    panelA_matrix,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=1
)


# Y labels
axA.set_yticks(
    np.arange(4)
)

axA.set_yticklabels(
    [
        "Attention",
        "Gradient × Input",
        "Integrated Gradients",
        "Adjusted consensus"
    ]
)

for label in axA.get_yticklabels():
    label.set_fontsize(10)
    label.set_fontweight("bold")


# X labels
axA.set_xticks(
    np.arange(L)
)

axA.set_xticklabels(
    [
        f"{aa}\n{i+1}"
        for i, aa in enumerate(
            representative_sequence
        )
    ]
)

for label in axA.get_xticklabels():
    label.set_fontsize(6.8)
    label.set_fontweight("bold")


axA.tick_params(
    axis="x",
    length=2,
    width=1,
    pad=2
)

axA.tick_params(
    axis="y",
    length=3,
    width=1,
    pad=4
)


axA.set_xlabel(
    "Residue and sequence position",
    fontsize=11,
    fontweight="bold",
    labelpad=5
)


# Representative sequence line
axA.text(
    0.5,
    1.045,

    f"Representative CPP: {TARGET_ID}   |   "
    f"{representative_sequence}",

    transform=axA.transAxes,

    ha="center",
    va="bottom",

    fontsize=9.7,
    fontweight="bold"
)


# Top-20% hotspot borders
n_hotspots = max(
    1,
    int(np.ceil(0.20 * L))
)

top_indices = np.argsort(
    consensus_values
)[-n_hotspots:]


for idx in top_indices:

    axA.add_patch(
        Rectangle(
            (
                idx - 0.5,
                2.5
            ),
            1,
            1,
            fill=False,
            edgecolor="white",
            linewidth=1.5
        )
    )


# Colorbar
cbarA = fig.colorbar(
    imA,
    ax=axA,
    fraction=0.014,
    pad=0.010,
    aspect=28
)

cbarA.set_label(
    "Normalized residue importance",
    fontsize=9,
    fontweight="bold",
    labelpad=5
)

cbarA.ax.tick_params(
    labelsize=8,
    width=1
)

for label in cbarA.ax.get_yticklabels():
    label.set_fontweight("bold")


# Panel letter
axA.text(
    -0.075,
    1.10,
    "A",

    transform=axA.transAxes,

    fontsize=20,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


fig.subplots_adjust(
    left=0.11,
    right=0.965,
    bottom=0.22,
    top=0.82
)


# Save
png_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.png"
)

pdf_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.pdf"
)

svg_A = (
    OUT_DIR
    / "Figure_3A_Representative_XAI_map.svg"
)


fig.savefig(
    png_A,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_A,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_A,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("Figure 3A saved:")
print(png_A)
print(pdf_A)
print(svg_A)

In [ ]:
# ======================================================================
# FIGURE 3B–D — BALANCED 2 × 2 MANUSCRIPT LAYOUT
#
#        B                         C
# XAI agreement           consensus stability
#
#        D                         D
# Internal heatmap          KELM heatmap
#
# All four plotting regions use equal width and equal row height
# ======================================================================

import matplotlib.pyplot as plt
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 10,

    "axes.labelsize": 11,
    "axes.labelweight": "bold",

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,

    "axes.linewidth": 1.1,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
#
# 2 × 2 grid with equal row/column dimensions
# ======================================================================

fig = plt.figure(
    figsize=(13.5, 7.2)
)


gs = fig.add_gridspec(

    2,
    2,

    width_ratios=[
        1,
        1
    ],

    height_ratios=[
        1,
        1
    ],

    left=0.085,
    right=0.94,

    bottom=0.10,
    top=0.93,

    wspace=0.24,
    hspace=0.34
)


axB  = fig.add_subplot(gs[0, 0])
axC  = fig.add_subplot(gs[0, 1])

axD1 = fig.add_subplot(gs[1, 0])
axD2 = fig.add_subplot(gs[1, 1])


# ======================================================================
# PANEL B
# ======================================================================

xB = np.arange(
    len(pairs)
)

widthB = 0.32


barsB1 = axB.bar(
    xB - widthB/2,
    internal_B,
    widthB,
    label="Internal test"
)

barsB2 = axB.bar(
    xB + widthB/2,
    kelm_B,
    widthB,
    label="KELM external"
)


axB.set_ylabel(
    "Mean sequence-level\nSpearman correlation",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


pretty_pairs = [

    "Attention\nvs\nGradient × Input",

    "Attention\nvs\nIntegrated Gradients",

    "Gradient × Input\nvs\nIntegrated Gradients"
]


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    pretty_pairs
)


for label in axB.get_xticklabels():

    label.set_fontsize(8.0)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# B axis: 0–1.05
# ----------------------------------------------------------------------

axB.set_ylim(
    0,
    1.05
)

axB.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


# ----------------------------------------------------------------------
# B annotations
# ----------------------------------------------------------------------

for bars in [
    barsB1,
    barsB2
]:

    for bar in bars:

        value = bar.get_height()

        if np.isnan(value):
            continue

        axB.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.018,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# B legend directly above axis
# ----------------------------------------------------------------------

axB.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.015
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.6,

    borderaxespad=0
)


axB.tick_params(
    axis="both",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axB.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# B letter — outside
# ----------------------------------------------------------------------

axB.text(

    -0.13,
    1.07,

    "B",

    transform=axB.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

categories_C = [

    "Overall rank\nSpearman",

    "Top-20% hotspot\nJaccard"
]


xC = np.arange(2)

widthC = 0.30


barsC1 = axC.bar(

    xC - widthC/2,

    internal_C,

    widthC,

    label="Internal test"
)


barsC2 = axC.bar(

    xC + widthC/2,

    kelm_C,

    widthC,

    label="KELM external"
)


axC.set_ylabel(
    "Agreement",
    fontsize=10.5,
    fontweight="bold",
    labelpad=5
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    categories_C
)


for label in axC.get_xticklabels():

    label.set_fontsize(8.2)
    label.set_fontweight("bold")


# ----------------------------------------------------------------------
# C axis: exactly same scale/height as B
# ----------------------------------------------------------------------

axC.set_ylim(
    0,
    1.05
)

axC.set_yticks(
    np.arange(
        0,
        1.01,
        0.20
    )
)


# ----------------------------------------------------------------------
# C annotations
# ----------------------------------------------------------------------

for bars in [
    barsC1,
    barsC2
]:

    for bar in bars:

        value = bar.get_height()

        axC.text(

            bar.get_x()
            + bar.get_width()/2,

            value + 0.018,

            f"{value:.3f}",

            ha="center",
            va="bottom",

            fontsize=8.5,
            fontweight="bold"
        )


# ----------------------------------------------------------------------
# C legend same location as B
# ----------------------------------------------------------------------

axC.legend(

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        1.015
    ),

    ncol=2,

    fontsize=8.5,

    handlelength=1.4,

    columnspacing=1.6,

    borderaxespad=0
)


axC.tick_params(
    axis="both",
    labelsize=9,
    width=1.1,
    length=4
)


for label in axC.get_yticklabels():
    label.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ----------------------------------------------------------------------
# C letter
# ----------------------------------------------------------------------

axC.text(

    -0.13,
    1.07,

    "C",

    transform=axC.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# PANEL D1 — INTERNAL TEST
#
# Same physical panel width/height as B
# ======================================================================

imD1 = axD1.imshow(

    internal_matrix.values,

    vmin=0,
    vmax=1,

    interpolation="nearest",

    # Fill exactly the subplot area
    aspect="auto"
)


axD1.set_xticks(
    np.arange(
        len(
            internal_matrix.columns
        )
    )
)

axD1.set_yticks(
    np.arange(
        len(
            internal_matrix.index
        )
    )
)


axD1.set_xticklabels(
    internal_matrix.columns
)

axD1.set_yticklabels(
    internal_matrix.index
)


for label in axD1.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD1.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD1,
    internal_matrix.values,
    fontsize=10
)


axD1.text(

    0.5,
    1.025,

    "Internal test",

    transform=axD1.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


axD1.tick_params(
    axis="both",
    width=1.0,
    length=3
)


# ======================================================================
# PANEL D2 — KELM EXTERNAL
#
# Same physical panel width/height as C
# ======================================================================

imD2 = axD2.imshow(

    kelm_matrix.values,

    vmin=0,
    vmax=1,

    interpolation="nearest",

    aspect="auto"
)


axD2.set_xticks(
    np.arange(
        len(
            kelm_matrix.columns
        )
    )
)

axD2.set_yticks(
    np.arange(
        len(
            kelm_matrix.index
        )
    )
)


axD2.set_xticklabels(
    kelm_matrix.columns
)

axD2.set_yticklabels(
    kelm_matrix.index
)


for label in axD2.get_xticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


for label in axD2.get_yticklabels():

    label.set_fontsize(9)
    label.set_fontweight("bold")


annotate_heatmap(
    axD2,
    kelm_matrix.values,
    fontsize=10
)


axD2.text(

    0.5,
    1.025,

    "KELM external",

    transform=axD2.transAxes,

    ha="center",
    va="bottom",

    fontsize=10.5,
    fontweight="bold"
)


axD2.tick_params(
    axis="both",
    width=1.0,
    length=3
)


# ======================================================================
# D LABEL
# ======================================================================

axD1.text(

    -0.13,
    1.07,

    "D",

    transform=axD1.transAxes,

    fontsize=19,
    fontweight="bold",

    ha="right",
    va="top",

    clip_on=False
)


# ======================================================================
# SHARED D COLORBAR
#
# Add manually OUTSIDE the 2 × 2 grid.
# Therefore it does not shrink or distort D2.
# ======================================================================

posD2 = axD2.get_position()


cbar_ax = fig.add_axes(
    [
        posD2.x1 + 0.015,    # x
        posD2.y0,            # y
        0.012,               # width
        posD2.height         # EXACT same height as D
    ]
)


cbarD = fig.colorbar(
    imD2,
    cax=cbar_ax
)


cbarD.set_label(

    "Hotspot Jaccard",

    fontsize=9.5,
    fontweight="bold",

    labelpad=5
)


cbarD.ax.tick_params(
    labelsize=8.5,
    width=1
)


for label in cbarD.ax.get_yticklabels():

    label.set_fontweight("bold")


# ======================================================================
# SAVE
# ======================================================================

png_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.png"
)

pdf_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.pdf"
)

svg_BD = (
    OUT_DIR
    / "Figure_3B-D_BALANCED_2x2.svg"
)


fig.savefig(
    png_BD,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_BD,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_BD,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\nFigure 3B-D generated:")
print(png_BD)
print(pdf_BD)
print(svg_BD)

In [ ]:
# ======================================================================
# FIGURE 4 — FINAL POLISHED VERSION
#
# A, B, C = top row
# D = full-width bottom
#
# Updates:
# - Panel D overlap fixed
# - D headings moved above boxes
# - More vertical spacing between sequence rows
# - D legend moved lower
# - A and C legends larger + bold
# ======================================================================

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 12,

    "axes.labelsize": 14,
    "axes.labelweight": "bold",

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    "legend.fontsize": 11,

    "axes.linewidth": 1.3,

    "lines.linewidth": 1.5,

    "savefig.dpi": 600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(17.5, 7.7)
)


outer = gridspec.GridSpec(
    2,
    1,

    figure=fig,

    height_ratios=[
        1.00,
        0.92
    ],

    hspace=0.22
)


# ======================================================================
# TOP ROW — A B C
# ======================================================================

top = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=outer[0],

    width_ratios=[
        1.00,
        1.05,
        1.10
    ],

    wspace=0.32
)


axA = fig.add_subplot(top[0])
axB = fig.add_subplot(top[1])
axC = fig.add_subplot(top[2])


# ======================================================================
# COMMON LEGEND
# ======================================================================

legend_handles = [

    Line2D(
        [0],
        [0],

        marker="o",
        linestyle="none",

        markerfacecolor="tab:blue",
        markeredgecolor="black",

        markersize=7.5,

        label="Internal test"
    ),

    Line2D(
        [0],
        [0],

        marker="s",
        linestyle="none",

        markerfacecolor="tab:orange",
        markeredgecolor="black",

        markersize=7.5,

        label="KELM external"
    )
]


# ======================================================================
# PANEL A
# ======================================================================

res_A = (
    res_plot
    .sort_values(
        internal_log2_col,
        ascending=True
    )
    .reset_index(drop=True)
)

yA = np.arange(len(res_A))


for i, row in res_A.iterrows():

    internal_value = float(
        row[internal_log2_col]
    )

    external_value = float(
        row[external_log2_col]
    )

    axA.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axA.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axA.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axA.axvline(
    0,
    linestyle="--",
    color="0.45",
    linewidth=1.2
)


axA.set_yticks(yA)

axA.set_yticklabels(
    res_A[residue_col].astype(str)
)


axA.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axA.set_ylabel(
    "Amino-acid residue",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axA.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axA.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axA.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# A legend — larger and bold
# ----------------------------------------------------------------------

legA = axA.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(0.55, 1.025),

    ncol=2,

    fontsize=11,

    handletextpad=0.45,

    columnspacing=1.4,

    borderaxespad=0
)

for txt in legA.get_texts():
    txt.set_fontweight("bold")


axA.text(
    0.03,
    0.025,
    "Depleted",
    transform=axA.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="bottom"
)

axA.text(
    0.97,
    0.025,
    "Enriched",
    transform=axA.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="right",
    va="bottom"
)


axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)


axA.text(
    -0.17,
    1.07,
    "A",
    transform=axA.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL B
# ======================================================================

xB = (
    res_plot[internal_log2_col]
    .astype(float)
    .to_numpy()
)

yB = (
    res_plot[external_log2_col]
    .astype(float)
    .to_numpy()
)

labelsB = (
    res_plot[residue_col]
    .astype(str)
    .to_numpy()
)


distance_from_center = (
    np.abs(xB)
    + np.abs(yB)
)

extreme_index = np.argmax(
    distance_from_center
)

normal_mask = np.ones(
    len(xB),
    dtype=bool
)

normal_mask[extreme_index] = False

normal_x = xB[normal_mask]
normal_y = yB[normal_mask]


axB.scatter(
    normal_x,
    normal_y,

    s=68,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7,

    zorder=3
)


for x, y, aa in zip(
    normal_x,
    normal_y,
    labelsB[normal_mask]
):

    axB.annotate(
        aa,
        xy=(x, y),
        xytext=(7, 5),
        textcoords="offset points",
        fontsize=10.5,
        fontweight="bold"
    )


all_normal = np.concatenate(
    [normal_x, normal_y]
)

plot_min = (
    np.min(all_normal)
    - 0.7
)

plot_max = (
    np.max(all_normal)
    + 0.7
)

plot_max = max(
    plot_max,
    0.8
)


axB.set_xlim(
    plot_min,
    plot_max
)

axB.set_ylim(
    plot_min,
    plot_max
)


axB.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],

    linestyle="--",

    color="0.45",

    linewidth=1.3
)


axB.axhline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)

axB.axvline(
    0,
    linestyle=":",
    color="0.65",
    linewidth=1
)


axB.set_xlabel(
    "Internal log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axB.set_ylabel(
    "KELM log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axB.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axB.get_yticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")


axB.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# B inset
# ----------------------------------------------------------------------

inset = inset_axes(
    axB,

    width="29%",
    height="29%",

    loc="upper left",

    bbox_to_anchor=(
        0.11,
        0.00,
        1,
        1
    ),

    bbox_transform=axB.transAxes,

    borderpad=0.7
)


extreme_x = xB[extreme_index]
extreme_y = yB[extreme_index]
extreme_label = labelsB[extreme_index]


inset.scatter(
    extreme_x,
    extreme_y,

    s=48,

    color="tab:blue",

    edgecolor="black",

    linewidth=0.7
)


inset.annotate(
    extreme_label,

    xy=(extreme_x, extreme_y),

    xytext=(5, 4),

    textcoords="offset points",

    fontsize=8.5,

    fontweight="bold"
)


extreme_min = min(
    extreme_x,
    extreme_y
)

extreme_max = max(
    extreme_x,
    extreme_y
)

extreme_pad = max(
    1.5,

    0.20
    * (
        extreme_max
        - extreme_min
    )
)


inset.set_xlim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)

inset.set_ylim(
    extreme_min - extreme_pad,
    extreme_max + extreme_pad
)


inset.plot(
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],
    [
        extreme_min - extreme_pad,
        extreme_max + extreme_pad
    ],

    linestyle="--",

    color="0.5",

    linewidth=0.8
)


inset.set_title(
    "Extreme effect",

    fontsize=8.5,

    fontweight="bold",

    pad=2
)


inset.tick_params(
    labelsize=7,
    width=0.8,
    length=2.5
)


for label in inset.get_xticklabels():
    label.set_fontweight("bold")

for label in inset.get_yticklabels():
    label.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


axB.text(
    -0.17,
    1.07,
    "B",
    transform=axB.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL C
# ======================================================================

motif_C = (
    motif_plot
    .sort_values(
        motif_internal_col,
        ascending=True
    )
    .reset_index(drop=True)
)

yC = np.arange(
    len(motif_C)
)


for i, row in motif_C.iterrows():

    internal_value = float(
        row[motif_internal_col]
    )

    external_value = float(
        row[motif_external_col]
    )

    axC.plot(
        [internal_value, external_value],
        [i, i],

        color="0.70",
        linewidth=1.4,

        zorder=1
    )

    axC.scatter(
        internal_value,
        i,

        s=64,

        marker="o",

        color="tab:blue",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )

    axC.scatter(
        external_value,
        i,

        s=64,

        marker="s",

        color="tab:orange",

        edgecolor="black",

        linewidth=0.7,

        zorder=3
    )


axC.set_yticks(yC)

axC.set_yticklabels(
    motif_C[motif_col]
    .astype(str)
)


axC.set_xlabel(
    "log$_2$ enrichment",
    fontsize=14,
    fontweight="bold",
    labelpad=5
)

axC.set_ylabel(
    "Replicated hotspot motif",
    fontsize=14,
    fontweight="bold",
    labelpad=7
)


for label in axC.get_xticklabels():
    label.set_fontsize(11)
    label.set_fontweight("bold")

for label in axC.get_yticklabels():
    label.set_fontsize(12)
    label.set_fontweight("bold")


axC.tick_params(
    width=1.2,
    length=5
)


# ----------------------------------------------------------------------
# C legend — larger and bold
# ----------------------------------------------------------------------

legC = axC.legend(
    handles=legend_handles,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(0.52, 1.025),

    ncol=2,

    fontsize=11,

    handletextpad=0.45,

    columnspacing=1.4,

    borderaxespad=0
)

for txt in legC.get_texts():
    txt.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


axC.text(
    -0.16,
    1.07,
    "C",
    transform=axC.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


# ======================================================================
# PANEL D — OVERLAP FIXED
# ======================================================================

axD = fig.add_subplot(
    outer[1]
)

axD.set_axis_off()


axD.text(
    -0.018,
    1.015,
    "D",
    transform=axD.transAxes,
    fontsize=23,
    fontweight="bold",
    ha="right",
    va="top",
    clip_on=False
)


rep_show = (
    rep_seq_df
    .head(3)
    .copy()
)


# ----------------------------------------------------------------------
# More vertical room between sequence rows
# ----------------------------------------------------------------------

y_positions = [
    0.68,
    0.37,
    0.06
]


for row_idx, (_, row) in enumerate(
    rep_show.iterrows()
):

    seq_id = str(
        row[seq_id_col]
    )

    seq = str(
        row[seq_col]
    )

    motif = (
        str(
            row[seq_motif_col]
        )
        if seq_motif_col is not None
        else ""
    )

    y0 = y_positions[row_idx]


    # --------------------------------------------------------------
    # Heading moved further above residue boxes
    # --------------------------------------------------------------

    axD.text(
        0.045,
        y0 + 0.135,

        f"{seq_id}   —   motif {motif}",

        transform=axD.transAxes,

        fontsize=12,

        fontweight="bold",

        ha="left",
        va="center"
    )


    n = len(seq)

    available_width = 0.92

    box_w = min(
        0.036,

        available_width
        / max(n, 1)
    )

    box_h = 0.10

    x_start = 0.045


    # --------------------------------------------------------------
    # Motif positions
    # --------------------------------------------------------------

    motif_positions = []

    if (
        motif
        and motif.lower() != "nan"
    ):

        start = 0

        while True:

            idx = seq.find(
                motif,
                start
            )

            if idx == -1:
                break

            motif_positions.extend(
                range(
                    idx,
                    idx + len(motif)
                )
            )

            start = idx + 1


    motif_positions = set(
        motif_positions
    )


    # --------------------------------------------------------------
    # Residue boxes
    # --------------------------------------------------------------

    for j, aa in enumerate(seq):

        x0 = (
            x_start
            + j * box_w
        )

        is_motif = (
            j in motif_positions
        )

        face = (
            "#e57373"
            if is_motif
            else "#e6e6e6"
        )


        rect = Rectangle(
            (
                x0,
                y0
            ),

            box_w * 0.94,
            box_h,

            transform=axD.transAxes,

            facecolor=face,

            edgecolor="black",

            linewidth=0.9
        )

        axD.add_patch(
            rect
        )


        axD.text(
            x0 + box_w * 0.47,
            y0 + box_h / 2,

            aa,

            transform=axD.transAxes,

            fontsize=9,

            fontweight="bold",

            ha="center",
            va="center"
        )


    # --------------------------------------------------------------
    # Motif stars
    # --------------------------------------------------------------

    for j in motif_positions:

        x0 = (
            x_start
            + j * box_w
        )

        axD.text(
            x0 + box_w * 0.47,
            y0 - 0.040,

            "★",

            transform=axD.transAxes,

            fontsize=8.5,

            ha="center",
            va="center"
        )


# ======================================================================
# PANEL D LEGEND — moved lower
# ======================================================================

legend_D = [

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e6e6e6",

        edgecolor="black",

        label="Other residue"
    ),

    Rectangle(
        (0, 0),
        1,
        1,

        facecolor="#e57373",

        edgecolor="black",

        label="Replicated motif"
    ),

    Line2D(
        [0],
        [0],

        marker="*",

        linestyle="none",

        color="black",

        markersize=10,

        label="Motif position"
    )
]


axD.legend(
    handles=legend_D,

    frameon=False,

    loc="lower center",

    bbox_to_anchor=(
        0.50,
        -0.10
    ),

    ncol=3,

    fontsize=11,

    columnspacing=2.5
)


# ======================================================================
# MARGINS
# ======================================================================

fig.subplots_adjust(
    left=0.065,
    right=0.985,
    bottom=0.09,
    top=0.93
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_4_Residue_Motif_Grammar_POLISHED_FINAL.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.04
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("POLISHED FIGURE 4 GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 6 — FINAL MANUSCRIPT VERSION WITH LARGER FONTS
#
# A = Pairwise CPP hotspot Jaccard matrices
# B = Cross-PLM hotspot-union support distribution
# C = CPP enrichment with increasing PLM agreement
# D = Sequence-level hotspot conservation
#
# Improvements:
# - larger fonts throughout
# - larger heatmap numbers
# - larger legends
# - larger bar-value labels
# - larger significance labels
# - larger panel letters
# - preserved gap between A and B
# - Panel B uses fraction_hotspot_union
# ======================================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


A_INTERNAL = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_internal_test_CPP.csv"
)

A_KELM = (
    TABLE_DIR
    / "hotspot_jaccard_matrix_kelm_external_CPP.csv"
)

B_FILE = (
    TABLE_DIR
    / "cross_plm_hotspot_support_distribution.csv"
)

C_FILE = (
    TABLE_DIR
    / "residue_level_cross_plm_conservation_enrichment.csv"
)

D_FILE = (
    SI_DIR
    / "cross_plm_hotspot_conservation_per_sequence.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(c)
        )
        .strip("_")
        .lower()
        for c in df.columns
    ]

    return df


def normalize_dataset(value):

    s = str(value).strip().lower()

    if "internal" in s:
        return "Internal"

    if (
        "kelm" in s
        or "external" in s
    ):
        return "KELM"

    return str(value)


def normalize_class(value):

    s = str(value).strip().lower()

    if (
        s in [
            "0",
            "0.0",
            "false",
            "negative"
        ]
        or "non_cpp" in s
        or "non-cpp" in s
        or "noncpp" in s
        or "non cpp" in s
    ):
        return "non-CPP"

    if (
        s in [
            "1",
            "1.0",
            "true",
            "positive"
        ]
        or s == "cpp"
    ):
        return "CPP"

    if s == "all":
        return "all"

    return str(value)


def style_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.1,
        length=4.5
    )

    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")

    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")


def first_existing(df, names):

    for name in names:

        if name in df.columns:
            return name

    return None


# ======================================================================
# STYLE — ENLARGED
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        12,

    "axes.labelsize":
        14,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        11,

    "ytick.labelsize":
        11,

    "legend.fontsize":
        10,

    "axes.linewidth":
        1.2,

    "savefig.dpi":
        600
})


# ======================================================================
# LOAD DATA
# ======================================================================

mat_internal = pd.read_csv(
    A_INTERNAL,
    index_col=0
)

mat_kelm = pd.read_csv(
    A_KELM,
    index_col=0
)

B = clean_columns(
    pd.read_csv(B_FILE)
)

C = clean_columns(
    pd.read_csv(C_FILE)
)

D = clean_columns(
    pd.read_csv(D_FILE)
)


# ======================================================================
# PRETTY MODEL LABELS
# ======================================================================

pretty_model = {

    "ESM2_320":
        "ESM2-320",

    "ESM2_640":
        "ESM2-640",

    "ESM2_1280":
        "ESM2-1280",

    "ProtT5":
        "ProtT5"
}


for matrix in [
    mat_internal,
    mat_kelm
]:

    matrix.index = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.index
    ]

    matrix.columns = [
        pretty_model.get(
            str(x),
            str(x)
        )
        for x in matrix.columns
    ]


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.8, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.075,
    right=0.975,

    bottom=0.09,
    top=0.97,

    # keep visible A–B separation
    wspace=0.39,

    hspace=0.31,

    width_ratios=[
        1.04,
        1.00
    ]
)


axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A — HEATMAPS
# ======================================================================

A_grid = gridspec.GridSpecFromSubplotSpec(
    1,
    3,

    subplot_spec=gs[0, 0],

    width_ratios=[
        1,
        1,
        0.045
    ],

    wspace=0.07
)


axA1 = fig.add_subplot(
    A_grid[0]
)

axA2 = fig.add_subplot(
    A_grid[1]
)

caxA = fig.add_subplot(
    A_grid[2]
)


imA = axA1.imshow(
    mat_internal.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


axA2.imshow(
    mat_kelm.values,

    vmin=0,
    vmax=1,

    cmap="Blues",

    interpolation="nearest",

    aspect="equal"
)


def format_heatmap(
    ax,
    matrix,
    show_y=True
):

    n = matrix.shape[0]

    ax.set_xticks(
        np.arange(n)
    )

    ax.set_yticks(
        np.arange(n)
    )


    ax.set_xticklabels(
        matrix.columns,

        rotation=35,

        ha="right",

        fontsize=10,

        fontweight="bold"
    )


    if show_y:

        ax.set_yticklabels(
            matrix.index,

            fontsize=10,

            fontweight="bold"
        )

    else:

        ax.set_yticklabels([])

        ax.tick_params(
            axis="y",
            length=0
        )


    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = float(
                matrix.iloc[i, j]
            )

            ax.text(
                j,
                i,

                f"{value:.2f}",

                ha="center",
                va="center",

                fontsize=9,

                fontweight="bold",

                color=(
                    "white"
                    if value > 0.55
                    else "black"
                )
            )


format_heatmap(
    axA1,
    mat_internal,
    True
)

format_heatmap(
    axA2,
    mat_kelm,
    False
)


axA1.text(
    0.5,
    1.035,

    "Internal test",

    transform=axA1.transAxes,

    ha="center",

    fontsize=12,

    fontweight="bold"
)


axA2.text(
    0.5,
    1.035,

    "KELM external",

    transform=axA2.transAxes,

    ha="center",

    fontsize=12,

    fontweight="bold"
)


cbA = fig.colorbar(
    imA,
    cax=caxA
)

cbA.set_label(
    "Mean per-sequence Jaccard",

    fontsize=11,

    fontweight="bold",

    labelpad=6
)

cbA.ax.tick_params(
    labelsize=9.5
)

for lab in cbA.ax.get_yticklabels():
    lab.set_fontweight("bold")


# ======================================================================
# PANEL B — HOTSPOT SUPPORT DISTRIBUTION
# ======================================================================

required_B = [
    "dataset",
    "class",
    "model_support_count",
    "fraction_hotspot_union"
]


missing_B = [
    c
    for c in required_B
    if c not in B.columns
]


if missing_B:

    raise KeyError(
        "Panel B missing columns:\n"
        + "\n".join(missing_B)
    )


B["dataset_clean"] = (
    B["dataset"]
    .map(normalize_dataset)
)

B["class_clean"] = (
    B["class"]
    .map(normalize_class)
)

B["support_clean"] = pd.to_numeric(
    B["model_support_count"],
    errors="coerce"
)

B["fraction_plot"] = pd.to_numeric(
    B["fraction_hotspot_union"],
    errors="coerce"
)


Bplot = B[
    B["support_clean"].isin(
        [1, 2, 3, 4]
    )
].copy()


Bplot = Bplot[
    Bplot["class_clean"].isin(
        [
            "CPP",
            "non-CPP"
        ]
    )
].copy()


group_order = [

    (
        "Internal",
        "CPP",
        "Internal CPP"
    ),

    (
        "Internal",
        "non-CPP",
        "Internal non-CPP"
    ),

    (
        "KELM",
        "CPP",
        "KELM CPP"
    ),

    (
        "KELM",
        "non-CPP",
        "KELM non-CPP"
    )
]


support_values = [
    1,
    2,
    3,
    4
]


xB = np.arange(
    4
)

widthB = 0.18


for group_index, (
    dataset_name,
    class_name,
    legend_name
) in enumerate(
    group_order
):

    values = []


    for support in support_values:

        row = Bplot[
            (
                Bplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Bplot["class_clean"]
                == class_name
            )
            &
            (
                Bplot["support_clean"]
                == support
            )
        ]


        if len(row):

            value = float(
                row["fraction_plot"]
                .iloc[0]
            )

        else:

            value = 0.0


        values.append(
            value
        )


    xpos = (
        xB
        + (
            group_index
            - 1.5
        )
        * widthB
    )


    bars = axB.bar(
        xpos,
        values,
        widthB,
        label=legend_name
    )


    # larger annotations
    for bar, value in zip(
        bars,
        values
    ):

        axB.text(
            bar.get_x()
            + bar.get_width() / 2,

            value + 0.010,

            f"{value:.2f}",

            ha="center",
            va="bottom",

            fontsize=9,

            fontweight="bold"
        )


axB.set_xticks(
    xB
)

axB.set_xticklabels(
    [
        "1 PLM",
        "2 PLMs",
        "3 PLMs",
        "4 PLMs"
    ],

    fontsize=11,

    fontweight="bold"
)


axB.set_ylabel(
    "Fraction of hotspot-union\nresidues",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axB
)


max_B = Bplot[
    "fraction_plot"
].max()

axB.set_ylim(
    0,
    max_B * 1.20
)


legB = axB.legend(
    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=9.5,

    columnspacing=1.1,

    handlelength=1.4
)


for text in legB.get_texts():
    text.set_fontweight("bold")


axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)


# ======================================================================
# PANEL C — CPP ENRICHMENT
# ======================================================================

dataset_col_C = first_existing(
    C,
    [
        "dataset",
        "dataset_name"
    ]
)


odds_col = first_existing(
    C,
    [
        "odds_ratio",
        "cpp_odds_ratio",
        "enrichment_odds_ratio",
        "or"
    ]
)


fdr_col = first_existing(
    C,
    [
        "fdr",
        "fdr_bh",
        "qvalue",
        "q_value",
        "adjusted_p",
        "adjusted_p_value"
    ]
)


threshold_col = first_existing(
    C,
    [
        "threshold",
        "support_threshold",
        "plm_support_threshold",
        "min_support",
        "minimum_support",
        "model_support_count",
        "support_count",
        "n_models",
        "support"
    ]
)


if threshold_col is None:

    for col in C.columns:

        lc = col.lower()

        if (
            "support" in lc
            or "threshold" in lc
            or "model" in lc
        ):

            test_vals = (
                C[col]
                .astype(str)
                .str.extract(
                    r"(\d+)"
                )[0]
            )

            nums = set(
                pd.to_numeric(
                    test_vals,
                    errors="coerce"
                )
                .dropna()
                .astype(int)
                .tolist()
            )

            if len(
                nums.intersection(
                    {2, 3, 4}
                )
            ) >= 2:

                threshold_col = col
                break


if odds_col is None:

    for col in C.columns:

        if (
            "odds" in col
            and "ratio" in col
        ):
            odds_col = col
            break


if odds_col is None:

    raise KeyError(
        "Panel C odds-ratio column not detected."
    )


Cplot = C.copy()


if dataset_col_C is not None:

    Cplot["dataset_clean"] = (
        Cplot[dataset_col_C]
        .map(normalize_dataset)
    )

else:

    if len(Cplot) == 6:

        Cplot["dataset_clean"] = (
            ["Internal"] * 3
            + ["KELM"] * 3
        )

    else:

        raise KeyError(
            "Could not identify Panel C dataset."
        )


def convert_support(value):

    s = str(value).lower()

    if "all" in s:
        return 4

    nums = re.findall(
        r"\d+",
        s
    )

    if nums:
        return int(
            nums[-1]
        )

    return np.nan


if threshold_col is not None:

    Cplot["support_num"] = (
        Cplot[threshold_col]
        .map(convert_support)
    )

else:

    Cplot["support_num"] = np.nan


    for dataset_name in [
        "Internal",
        "KELM"
    ]:

        idx = Cplot[
            Cplot["dataset_clean"]
            == dataset_name
        ].index.tolist()


        if len(idx) == 3:

            Cplot.loc[
                idx,
                "support_num"
            ] = [
                2,
                3,
                4
            ]

        else:

            raise KeyError(
                "Could not infer Panel C support levels."
            )


Cplot["odds_plot"] = pd.to_numeric(
    Cplot[odds_col],
    errors="coerce"
)


levels_C = [
    2,
    3,
    4
]

labels_C = [
    "≥2 PLMs",
    "≥3 PLMs",
    "All 4 PLMs"
]


xC = np.arange(
    3
)

widthC = 0.32


for di, dataset_name in enumerate(
    [
        "Internal",
        "KELM"
    ]
):

    vals = []
    ps = []


    for support in levels_C:

        row = Cplot[
            (
                Cplot["dataset_clean"]
                == dataset_name
            )
            &
            (
                Cplot["support_num"]
                == support
            )
        ]


        if len(row):

            vals.append(
                float(
                    row["odds_plot"]
                    .iloc[0]
                )
            )


            if fdr_col is not None:

                ps.append(
                    float(
                        row[fdr_col]
                        .iloc[0]
                    )
                )

            else:

                ps.append(
                    np.nan
                )

        else:

            vals.append(
                np.nan
            )

            ps.append(
                np.nan
            )


    xpos = (
        xC
        + (
            -widthC / 2
            if di == 0
            else widthC / 2
        )
    )


    bars = axC.bar(
        xpos,
        vals,
        widthC,

        label=(
            "Internal test"
            if dataset_name == "Internal"
            else "KELM external"
        )
    )


    for bar, val, p in zip(
        bars,
        vals,
        ps
    ):

        if np.isnan(val):
            continue


        # larger value label
        axC.text(
            bar.get_x()
            + bar.get_width() / 2,

            val + 0.05,

            f"{val:.2f}",

            ha="center",
            va="bottom",

            fontsize=9.5,

            fontweight="bold"
        )


        if not np.isnan(p):

            if p < 0.001:
                sig = "***"

            elif p < 0.01:
                sig = "**"

            elif p < 0.05:
                sig = "*"

            else:
                sig = "ns"


            axC.text(
                bar.get_x()
                + bar.get_width() / 2,

                val + 0.22,

                sig,

                ha="center",
                va="bottom",

                fontsize=10,

                fontweight="bold"
            )


axC.axhline(
    1,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    labels_C,

    fontsize=11,

    fontweight="bold"
)


axC.set_ylabel(
    "CPP enrichment\nodds ratio",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axC
)


legC = axC.legend(
    frameon=False,

    loc="upper left",

    fontsize=9.5
)


for text in legC.get_texts():
    text.set_fontweight("bold")


axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)


# ======================================================================
# PANEL D — SEQUENCE-LEVEL CONSERVATION
# ======================================================================

dataset_col_D = first_existing(
    D,
    [
        "dataset",
        "dataset_name"
    ]
)

label_col_D = first_existing(
    D,
    [
        "label",
        "class",
        "cpp_label"
    ]
)


jaccard_col = first_existing(
    D,
    [
        "mean_pairwise_jaccard",
        "mean_jaccard",
        "pairwise_jaccard"
    ]
)

support3_col = first_existing(
    D,
    [
        "fraction_supported_by_at_least_3",
        "fraction_supported_by_3",
        "fraction_at_least_3"
    ]
)

support4_col = first_existing(
    D,
    [
        "fraction_supported_by_all_4",
        "fraction_supported_by_4",
        "fraction_all_4"
    ]
)


if jaccard_col is None:

    for col in D.columns:

        if "jaccard" in col:
            jaccard_col = col
            break


if support3_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "3" in col
        ):
            support3_col = col
            break


if support4_col is None:

    for col in D.columns:

        if (
            "fraction" in col
            and "4" in col
        ):
            support4_col = col
            break


if any(
    x is None
    for x in [
        dataset_col_D,
        label_col_D,
        jaccard_col,
        support3_col,
        support4_col
    ]
):

    raise KeyError(
        "Panel D column detection failed."
    )


D["dataset_clean"] = (
    D[dataset_col_D]
    .map(normalize_dataset)
)

D["class_clean"] = (
    D[label_col_D]
    .map(normalize_class)
)


metrics_D = [

    (
        jaccard_col,
        "Mean pairwise\nJaccard"
    ),

    (
        support3_col,
        "Supported by\n≥3 PLMs"
    ),

    (
        support4_col,
        "Supported by\nall 4 PLMs"
    )
]


groups_D = [

    (
        "Internal",
        "non-CPP"
    ),

    (
        "Internal",
        "CPP"
    ),

    (
        "KELM",
        "non-CPP"
    ),

    (
        "KELM",
        "CPP"
    )
]


base_positions = np.arange(
    3
)

offsets_D = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


for gi, (
    dataset_name,
    class_name
) in enumerate(
    groups_D
):

    group = D[
        (
            D["dataset_clean"]
            == dataset_name
        )
        &
        (
            D["class_clean"]
            == class_name
        )
    ]


    vals = [

        pd.to_numeric(
            group[col],
            errors="coerce"
        )
        .dropna()
        .to_numpy()

        for col, _
        in metrics_D
    ]


    axD.boxplot(
        vals,

        positions=(
            base_positions
            + offsets_D[gi]
        ),

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.5
        ),

        boxprops=dict(
            facecolor=(
                "0.82"
                if class_name == "CPP"
                else "white"
            ),

            edgecolor="black",

            linewidth=0.9
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.9
        ),

        capprops=dict(
            color="black",
            linewidth=0.9
        )
    )


axD.set_xticks(
    base_positions
)

axD.set_xticklabels(
    [
        label
        for _,
        label in metrics_D
    ],

    fontsize=10.5,

    fontweight="bold"
)


axD.set_ylabel(
    "Sequence-level\nconservation",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


style_ticks(
    axD
)


legend_D = [

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="black",
        label="Internal CPP"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.45",
        label="KELM non-CPP"
    ),

    Patch(
        facecolor="0.82",
        edgecolor="0.45",
        label="KELM CPP"
    )
]


legD = axD.legend(
    handles=legend_D,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=9.2
)


for text in legD.get_texts():
    text.set_fontweight("bold")


axD.spines["top"].set_visible(False)
axD.spines["right"].set_visible(False)


# ======================================================================
# PANEL LETTERS — LARGER
# ======================================================================

def add_letter(
    ax,
    letter,
    x=-0.11,
    y=1.03
):

    ax.text(
        x,
        y,

        letter,

        transform=ax.transAxes,

        fontsize=23,

        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,

        zorder=100
    )


add_letter(
    axA1,
    "A",
    x=-0.18
)

add_letter(
    axB,
    "B",
    x=-0.13
)

add_letter(
    axC,
    "C"
)

add_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_6_Cross_PLM_Hotspot_Conservation_LARGE_FONT.svg"
)


fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    pad_inches=0.05
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    pad_inches=0.05
)


plt.show()
plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 6 WITH LARGER FONTS GENERATED")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ======================================================================
# FIGURE 7 — FINAL MANUSCRIPT VERSION
# Physicochemical and Positional Organization of Consensus CPP Hotspots
#
# A = Replicated physicochemical enrichment
# B = Sequence-level charge and hydropathy of hotspots vs non-hotspots
# C = Regional positional enrichment of CPP hotspots
# D = Detailed 10-bin positional profile
#
# Uses exact source-table columns provided above.
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

TABLE_DIR = (
    PROJECT
    / "07_results"
    / "tables_main"
)

SI_DIR = (
    PROJECT
    / "07_results"
    / "tables_SI"
)

OUT_DIR = (
    PROJECT
    / "07_results"
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


A_FILE = (
    TABLE_DIR
    / "physicochemical_enrichment_replication.csv"
)

B_FILE = (
    SI_DIR
    / "physicochemical_properties_per_sequence.csv"
)

C_FILE = (
    TABLE_DIR
    / "hotspot_positional_region_enrichment.csv"
)

D_FILE = (
    TABLE_DIR
    / "hotspot_position_10bin_distribution.csv"
)


# ======================================================================
# LOAD DATA
# ======================================================================

A = pd.read_csv(A_FILE)
B = pd.read_csv(B_FILE)
C = pd.read_csv(C_FILE)
D = pd.read_csv(D_FILE)


print("=" * 90)
print("FIGURE 7 SOURCE TABLES LOADED")
print("=" * 90)

print("\nA:", A.shape)
print("B:", B.shape)
print("C:", C.shape)
print("D:", D.shape)


# ======================================================================
# HELPERS
# ======================================================================

def normalize_dataset(value):

    s = str(value).lower()

    if "internal" in s:
        return "Internal"

    if (
        "kelm" in s
        or "external" in s
    ):
        return "KELM"

    return str(value)


def significance_symbol(p):

    if pd.isna(p):
        return ""

    if p < 0.001:
        return "***"

    if p < 0.01:
        return "**"

    if p < 0.05:
        return "*"

    return "ns"


def bold_ticks(ax):

    ax.tick_params(
        axis="both",
        width=1.1,
        length=4.5
    )

    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")


def clean_axes(ax):

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_letter(
    ax,
    letter,
    x=-0.11,
    y=1.03
):

    ax.text(
        x,
        y,
        letter,

        transform=ax.transAxes,

        fontsize=23,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100
    )


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        12,

    "axes.labelsize":
        13,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        10.5,

    "ytick.labelsize":
        10.5,

    "legend.fontsize":
        9.5,

    "axes.linewidth":
        1.2,

    "savefig.dpi":
        600
})


# ======================================================================
# FIGURE
# ======================================================================

fig = plt.figure(
    figsize=(14.5, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.080,
    right=0.975,

    bottom=0.095,
    top=0.970,

    wspace=0.30,
    hspace=0.34
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANEL A
# PHYSICOCHEMICAL ENRICHMENT
# ======================================================================

Aplot = A.copy()


property_order = [
    "basic_positive",
    "acidic_negative",
    "polar_uncharged",
    "aromatic",
    "aliphatic_hydrophobic",
    "structure_special"
]


property_labels = {
    "basic_positive":
        "Basic\npositive",

    "acidic_negative":
        "Acidic\nnegative",

    "polar_uncharged":
        "Polar\nuncharged",

    "aromatic":
        "Aromatic",

    "aliphatic_hydrophobic":
        "Aliphatic\nhydrophobic",

    "structure_special":
        "Structure\nspecial"
}


Aplot = (
    Aplot
    .set_index("property_class")
    .loc[property_order]
    .reset_index()
)


xA = np.arange(
    len(property_order)
)

widthA = 0.34


bars_internal = axA.bar(
    xA - widthA/2,

    Aplot[
        "internal_log2_enrichment"
    ],

    widthA,

    label="Internal test"
)


bars_kelm = axA.bar(
    xA + widthA/2,

    Aplot[
        "kelm_log2_enrichment"
    ],

    widthA,

    label="KELM external"
)


# zero reference
axA.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


# ----------------------------------------------------------------------
# Significance labels
# Put above positive bars / below negative bars
# ----------------------------------------------------------------------

def annotate_significance(
    ax,
    bars,
    pvalues
):

    for bar, p in zip(
        bars,
        pvalues
    ):

        h = bar.get_height()

        sig = significance_symbol(
            p
        )


        if h >= 0:

            y = h + 0.10

            va = "bottom"

        else:

            y = h - 0.11

            va = "top"


        ax.text(
            bar.get_x()
            + bar.get_width()/2,

            y,

            sig,

            ha="center",
            va=va,

            fontsize=9.5,

            fontweight="bold"
        )


annotate_significance(
    axA,
    bars_internal,
    Aplot["internal_fdr"]
)

annotate_significance(
    axA,
    bars_kelm,
    Aplot["kelm_fdr"]
)


axA.set_xticks(
    xA
)

axA.set_xticklabels(
    [
        property_labels[x]
        for x in property_order
    ],

    fontsize=9.8,

    fontweight="bold"
)


axA.set_ylabel(
    "log$_2$ enrichment\nin CPP hotspots",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legA = axA.legend(
    frameon=False,

    loc="lower left",

    fontsize=9.5
)

for text in legA.get_texts():
    text.set_fontweight("bold")


bold_ticks(axA)
clean_axes(axA)

add_panel_letter(
    axA,
    "A"
)


# ======================================================================
# PANEL B
# CHARGE SCORE + HYDROPATHY
#
# CPP sequences only
# ======================================================================

Bplot = B.copy()


# CPP only
Bplot = Bplot[
    Bplot["label"] == 1
].copy()


Bplot["dataset_clean"] = (
    Bplot["dataset"]
    .map(normalize_dataset)
)


# ----------------------------------------------------------------------
# Boxplot groups:
#
# Metric 1: charge
#   Internal hotspot
#   Internal non-hotspot
#   KELM hotspot
#   KELM non-hotspot
#
# Metric 2: hydropathy
#   same four groups
# ----------------------------------------------------------------------

metrics_B = [

    (
        "hotspot_charge_score",
        "nonhotspot_charge_score",
        "Mean charge\nscore"
    ),

    (
        "hotspot_hydropathy_KD",
        "nonhotspot_hydropathy_KD",
        "Mean hydropathy"
    )
]


base_B = np.array([
    0,
    1
])


offsets_B = [
    -0.27,
    -0.09,
    0.09,
    0.27
]


group_specs_B = [

    (
        "Internal",
        "hotspot",
        "Internal hotspot",
        "0.55"
    ),

    (
        "Internal",
        "nonhotspot",
        "Internal non-hotspot",
        "white"
    ),

    (
        "KELM",
        "hotspot",
        "KELM hotspot",
        "0.72"
    ),

    (
        "KELM",
        "nonhotspot",
        "KELM non-hotspot",
        "white"
    )
]


for gi, (
    dataset_name,
    region_type,
    legend_name,
    face
) in enumerate(
    group_specs_B
):

    metric_values = []


    for hotspot_col, nonhot_col, _ in metrics_B:

        col = (
            hotspot_col
            if region_type == "hotspot"
            else nonhot_col
        )


        values = (
            Bplot[
                Bplot["dataset_clean"]
                == dataset_name
            ][col]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        metric_values.append(
            values
        )


    axB.boxplot(
        metric_values,

        positions=(
            base_B
            + offsets_B[gi]
        ),

        widths=0.16,

        patch_artist=True,

        showfliers=False,

        medianprops=dict(
            color="tab:orange",
            linewidth=1.5
        ),

        boxprops=dict(
            facecolor=face,
            edgecolor="black",
            linewidth=0.9
        ),

        whiskerprops=dict(
            color="black",
            linewidth=0.9
        ),

        capprops=dict(
            color="black",
            linewidth=0.9
        )
    )


axB.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


axB.set_xticks(
    base_B
)

axB.set_xticklabels(
    [
        "Mean charge\nscore",
        "Mean hydropathy"
    ],

    fontsize=10,

    fontweight="bold"
)


axB.set_ylabel(
    "Sequence-level\nproperty value",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legend_B = [

    Patch(
        facecolor="0.55",
        edgecolor="black",
        label="Internal hotspot"
    ),

    Patch(
        facecolor="white",
        edgecolor="black",
        label="Internal non-hotspot"
    ),

    Patch(
        facecolor="0.72",
        edgecolor="black",
        label="KELM hotspot"
    ),

    Patch(
        facecolor="white",
        edgecolor="0.45",
        label="KELM non-hotspot"
    )
]


legB = axB.legend(
    handles=legend_B,

    frameon=False,

    ncol=2,

    loc="upper right",

    fontsize=8.7,

    columnspacing=1.0,

    handlelength=1.5
)


for text in legB.get_texts():
    text.set_fontweight("bold")


bold_ticks(axB)
clean_axes(axB)

add_panel_letter(
    axB,
    "B"
)


# ======================================================================
# PANEL C
# REGIONAL ORGANIZATION
#
# CPP only
# ======================================================================

Cplot = C[
    C["class"] == "CPP"
].copy()


Cplot["dataset_clean"] = (
    Cplot["dataset"]
    .map(normalize_dataset)
)


region_order = [
    "N_terminal",
    "Middle",
    "C_terminal"
]


region_labels = {
    "N_terminal":
        "N-terminal",

    "Middle":
        "Middle",

    "C_terminal":
        "C-terminal"
}


xC = np.arange(
    3
)

widthC = 0.34


values_internal = []
values_kelm = []

fdr_internal = []
fdr_kelm = []


for region in region_order:

    row_i = Cplot[
        (
            Cplot["dataset_clean"]
            == "Internal"
        )
        &
        (
            Cplot["region"]
            == region
        )
    ]


    row_k = Cplot[
        (
            Cplot["dataset_clean"]
            == "KELM"
        )
        &
        (
            Cplot["region"]
            == region
        )
    ]


    values_internal.append(
        float(
            row_i[
                "log2_enrichment"
            ].iloc[0]
        )
    )

    values_kelm.append(
        float(
            row_k[
                "log2_enrichment"
            ].iloc[0]
        )
    )


    fdr_internal.append(
        float(
            row_i[
                "fdr_bh"
            ].iloc[0]
        )
    )

    fdr_kelm.append(
        float(
            row_k[
                "fdr_bh"
            ].iloc[0]
        )
    )


bars_C_i = axC.bar(
    xC - widthC/2,

    values_internal,

    widthC,

    label="Internal test"
)


bars_C_k = axC.bar(
    xC + widthC/2,

    values_kelm,

    widthC,

    label="KELM external"
)


axC.axhline(
    0,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


annotate_significance(
    axC,
    bars_C_i,
    fdr_internal
)

annotate_significance(
    axC,
    bars_C_k,
    fdr_kelm
)


axC.set_xticks(
    xC
)

axC.set_xticklabels(
    [
        region_labels[x]
        for x in region_order
    ],

    fontsize=10.5,

    fontweight="bold"
)


axC.set_ylabel(
    "log$_2$ enrichment\nin CPP hotspots",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


legC = axC.legend(
    frameon=False,

    loc="lower left",

    fontsize=9.5
)


for text in legC.get_texts():
    text.set_fontweight("bold")


bold_ticks(axC)
clean_axes(axC)

add_panel_letter(
    axC,
    "C"
)


# ======================================================================
# PANEL D
# DETAILED TEN-BIN POSITIONAL PROFILE
#
# CPP only
# ======================================================================

Dplot = D[
    D["class"] == "CPP"
].copy()


Dplot["dataset_clean"] = (
    Dplot["dataset"]
    .map(normalize_dataset)
)


internal_D = (
    Dplot[
        Dplot["dataset_clean"]
        == "Internal"
    ]
    .sort_values(
        "position_bin"
    )
)


kelm_D = (
    Dplot[
        Dplot["dataset_clean"]
        == "KELM"
    ]
    .sort_values(
        "position_bin"
    )
)


xD = np.arange(
    1,
    11
)


# ----------------------------------------------------------------------
# Central region shading: 20–70%
# corresponds approximately bins 3–7
# ----------------------------------------------------------------------

axD.axvspan(
    2.5,
    7.5,

    alpha=0.12,

    color="tab:blue",

    zorder=0
)


axD.plot(
    xD,

    internal_D[
        "hotspot_to_nonhotspot_ratio"
    ].to_numpy(),

    marker="o",

    linewidth=1.8,

    markersize=5.5,

    label="Internal test"
)


axD.plot(
    xD,

    kelm_D[
        "hotspot_to_nonhotspot_ratio"
    ].to_numpy(),

    marker="s",

    linewidth=1.8,

    markersize=5.5,

    label="KELM external"
)


# neutral reference
axD.axhline(
    1,

    linestyle="--",

    linewidth=1.0,

    color="0.45"
)


position_labels = [
    "0–10",
    "10–20",
    "20–30",
    "30–40",
    "40–50",
    "50–60",
    "60–70",
    "70–80",
    "80–90",
    "90–100"
]


axD.set_xticks(
    xD
)

axD.set_xticklabels(
    position_labels,

    rotation=35,

    ha="right",

    fontsize=9.5,

    fontweight="bold"
)


axD.set_xlabel(
    "Relative sequence position (%)",

    fontsize=12.5,

    fontweight="bold",

    labelpad=6
)


axD.set_ylabel(
    "Hotspot/non-hotspot\nfrequency ratio",

    fontsize=13,

    fontweight="bold",

    labelpad=7
)


# small central-region annotation
ymax_D = max(
    internal_D[
        "hotspot_to_nonhotspot_ratio"
    ].max(),

    kelm_D[
        "hotspot_to_nonhotspot_ratio"
    ].max()
)


axD.text(
    5.0,
    ymax_D * 0.97,

    "Central region",

    ha="center",
    va="top",

    fontsize=9.5,

    fontweight="bold"
)


legD = axD.legend(
    frameon=False,

    loc="upper right",

    fontsize=9.5
)


for text in legD.get_texts():
    text.set_fontweight("bold")


bold_ticks(axD)
clean_axes(axD)

add_panel_letter(
    axD,
    "D"
)


# ======================================================================
# SAVE
# ======================================================================

png_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.png"
)

pdf_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.pdf"
)

svg_file = (
    OUT_DIR
    / "Figure_7_Physicochemical_and_Positional_Grammar_FINAL.svg"
)


fig.savefig(
    png_file,

    dpi=600,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    pdf_file,

    bbox_inches="tight",

    pad_inches=0.05
)


fig.savefig(
    svg_file,

    bbox_inches="tight",

    pad_inches=0.05
)


plt.show()

plt.close(fig)


print("\n" + "=" * 90)
print("FIGURE 7 GENERATED SUCCESSFULLY")
print("=" * 90)

print("\nPNG:")
print(png_file)

print("\nPDF:")
print(pdf_file)

print("\nSVG:")
print(svg_file)

In [ ]:
# ============================================================
# FIGURE 4E
# Composition-controlled motif enrichment
#
# Grouped bars:
#   Internal vs KELM
# Y-axis:
#   Fold over composition-preserving null
#
# Significance:
#   *  FDR < 0.05
#
# Output:
#   Figure4E_Composition_Controlled_Motifs.png
#   Figure4E_Composition_Controlled_Motifs.pdf
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/pLM4CPP_XAI_2026")

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "motif_composition_controlled_null.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_main"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PNG_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.png"
)

PDF_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.pdf"
)

print("Input exists:", INPUT_FILE.exists())
print("Input:", INPUT_FILE)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

motif_order = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]

df["motif"] = pd.Categorical(
    df["motif"],
    categories=motif_order,
    ordered=True
)

df = df.sort_values(
    ["motif", "dataset"]
)

# ------------------------------------------------------------
# SPLIT DATASETS
# ------------------------------------------------------------

internal = (
    df[df["dataset"] == "internal_test"]
    .set_index("motif")
    .loc[motif_order]
)

kelm = (
    df[df["dataset"] == "kelm_external"]
    .set_index("motif")
    .loc[motif_order]
)

internal_fold = internal["fold_over_null"].to_numpy()
kelm_fold = kelm["fold_over_null"].to_numpy()

internal_fdr = internal["fdr"].to_numpy()
kelm_fdr = kelm["fdr"].to_numpy()

# ------------------------------------------------------------
# PLOT SETTINGS
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.2,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(
    figsize=(6.2, 4.2)
)

x = np.arange(
    len(motif_order)
)

bar_width = 0.36

# ------------------------------------------------------------
# BARS
# ------------------------------------------------------------

bars_internal = ax.bar(
    x - bar_width / 2,
    internal_fold,
    width=bar_width,
    label="Internal",
    edgecolor="black",
    linewidth=1.0,
)

bars_kelm = ax.bar(
    x + bar_width / 2,
    kelm_fold,
    width=bar_width,
    label="KELM",
    edgecolor="black",
    linewidth=1.0,
)

# ------------------------------------------------------------
# REFERENCE LINE
# ------------------------------------------------------------

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1.2,
    color="black",
)

# ------------------------------------------------------------
# SIGNIFICANCE STARS
# ------------------------------------------------------------

def add_sig_stars(
    bars,
    fdr_values,
):

    for bar, fdr in zip(
        bars,
        fdr_values
    ):

        if fdr < 0.001:
            star = "***"
        elif fdr < 0.01:
            star = "**"
        elif fdr < 0.05:
            star = "*"
        else:
            star = ""

        if star:

            height = (
                bar.get_height()
            )

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,
                height + 0.07,
                star,
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

add_sig_stars(
    bars_internal,
    internal_fdr
)

add_sig_stars(
    bars_kelm,
    kelm_fdr
)

# ------------------------------------------------------------
# VALUE LABELS
# ------------------------------------------------------------

def add_value_labels(
    bars,
):

    for bar in bars:

        height = (
            bar.get_height()
        )

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.015,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            rotation=90,
        )

add_value_labels(
    bars_internal
)

add_value_labels(
    bars_kelm
)

# ------------------------------------------------------------
# AXES
# ------------------------------------------------------------

ax.set_xticks(
    x
)

ax.set_xticklabels(
    motif_order,
    fontweight="bold"
)

ax.set_ylabel(
    "Fold over composition-preserving null",
    fontweight="bold"
)

ax.set_xlabel(
    "Motif",
    fontweight="bold"
)

ax.set_ylim(
    0,
    max(
        internal_fold.max(),
        kelm_fold.max()
    ) + 0.55
)

# ------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------

legend = ax.legend(
    frameon=False,
    loc="upper right",
    prop={
        "weight": "bold",
        "size": 9,
    },
)

# ------------------------------------------------------------
# CLEAN STYLE
# ------------------------------------------------------------

ax.spines["top"].set_visible(
    False
)

ax.spines["right"].set_visible(
    False
)

ax.tick_params(
    width=1.2
)

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    alpha=0.5
)

# ------------------------------------------------------------
# PANEL LABEL
# ------------------------------------------------------------

ax.text(
    -0.11,
    1.03,
    "E",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

plt.tight_layout()

plt.savefig(
    PNG_FILE,
    dpi=600,
    bbox_inches="tight",
)

plt.savefig(
    PDF_FILE,
    bbox_inches="tight",
)

plt.show()

print("\nSaved:")
print(PNG_FILE)
print(PDF_FILE)

In [ ]:
# ============================================================
# FIGURE 4E — FINAL COMPACT VERSION
# Composition-controlled motif enrichment
#
# Plot:
#   Internal vs KELM
#
# Y-axis:
#   Fold over composition-preserving null
#
# Significance:
#   *   FDR < 0.05
#   **  FDR < 0.01
#   *** FDR < 0.001
#
# Saves:
#   PNG (600 dpi)
#   PDF
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "motif_composition_controlled_null.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_main"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PNG_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.png"
)

PDF_FILE = (
    OUTPUT_DIR
    / "Figure4E_Composition_Controlled_Motifs.pdf"
)


print("Input file:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())

assert INPUT_FILE.exists(), (
    f"Input file not found: {INPUT_FILE}"
)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# MOTIF ORDER
# ============================================================

motif_order = [
    "RR",
    "WK",
    "WR",
    "RW",
    "RL",
    "LR",
]


# ============================================================
# PREPARE DATA
# ============================================================

internal = (
    df[
        df["dataset"] == "internal_test"
    ]
    .set_index("motif")
    .loc[motif_order]
)

kelm = (
    df[
        df["dataset"] == "kelm_external"
    ]
    .set_index("motif")
    .loc[motif_order]
)


internal_fold = (
    internal[
        "fold_over_null"
    ]
    .to_numpy(
        dtype=float
    )
)

kelm_fold = (
    kelm[
        "fold_over_null"
    ]
    .to_numpy(
        dtype=float
    )
)

internal_fdr = (
    internal[
        "fdr"
    ]
    .to_numpy(
        dtype=float
    )
)

kelm_fdr = (
    kelm[
        "fdr"
    ]
    .to_numpy(
        dtype=float
    )
)


# ============================================================
# OPTIONAL CHECK
# ============================================================

check_table = pd.DataFrame(
    {
        "Motif": motif_order,
        "Internal_fold": internal_fold,
        "Internal_FDR": internal_fdr,
        "KELM_fold": kelm_fold,
        "KELM_FDR": kelm_fdr,
    }
)

print("\nData used for Figure 4E:")
display(
    check_table
)


# ============================================================
# GLOBAL STYLE
# ============================================================

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "font.weight": "bold",

        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.linewidth": 1.2,

        "xtick.labelsize": 10,
        "ytick.labelsize": 10,

        "legend.fontsize": 9,
    }
)


# ============================================================
# CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(6.8, 4.8)
)


x = np.arange(
    len(motif_order)
)

bar_width = 0.35


# ============================================================
# BARS
# ============================================================

bars_internal = ax.bar(
    x - bar_width / 2,
    internal_fold,
    width=bar_width,
    label="Internal",
    edgecolor="black",
    linewidth=1.0,
)

bars_kelm = ax.bar(
    x + bar_width / 2,
    kelm_fold,
    width=bar_width,
    label="KELM",
    edgecolor="black",
    linewidth=1.0,
)


# ============================================================
# NULL EXPECTATION
# ============================================================

ax.axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1.2,
    zorder=0,
)


# ============================================================
# SIGNIFICANCE FUNCTION
# ============================================================

def get_significance_label(
    fdr
):

    if fdr < 0.001:
        return "***"

    elif fdr < 0.01:
        return "**"

    elif fdr < 0.05:
        return "*"

    else:
        return ""


# ============================================================
# ADD SIGNIFICANCE STARS
# ============================================================

for bar, fdr in zip(
    bars_internal,
    internal_fdr
):

    sig = (
        get_significance_label(
            fdr
        )
    )

    if sig:

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + 0.055,

            sig,

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold",
        )


for bar, fdr in zip(
    bars_kelm,
    kelm_fdr
):

    sig = (
        get_significance_label(
            fdr
        )
    )

    if sig:

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + 0.055,

            sig,

            ha="center",
            va="bottom",

            fontsize=11,
            fontweight="bold",
        )


# ============================================================
# AXES
# ============================================================

ax.set_xticks(
    x
)

ax.set_xticklabels(
    motif_order,
    fontweight="bold",
)


ax.set_xlabel(
    "Motif",
    fontweight="bold",
    fontsize=10,
)


ax.set_ylabel(
    "Fold over composition-preserving null",
    fontweight="bold",
    fontsize=10,
)


ax.set_ylim(
    0,
    2.55
)


# ============================================================
# FORCE TICK LABELS BOLD
# ============================================================

for label in ax.get_xticklabels():
    label.set_fontweight(
        "bold"
    )

for label in ax.get_yticklabels():
    label.set_fontweight(
        "bold"
    )


# ============================================================
# LEGEND
# ============================================================

legend = ax.legend(
    frameon=False,
    loc="upper right",
)

for text in legend.get_texts():
    text.set_fontweight(
        "bold"
    )


# ============================================================
# CLEAN ACS-LIKE STYLE
# ============================================================

ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)

ax.spines[
    "left"
].set_linewidth(
    1.2
)

ax.spines[
    "bottom"
].set_linewidth(
    1.2
)


ax.tick_params(
    axis="both",
    width=1.2,
    length=4,
)


# very subtle horizontal grid
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.5,
    alpha=0.25,
)


# ============================================================
# PANEL LABEL
# ============================================================

ax.text(
    -0.105,
    1.03,
    "E",

    transform=ax.transAxes,

    fontsize=16,
    fontweight="bold",

    ha="left",
    va="top",
)


# ============================================================
# LAYOUT
# ============================================================

plt.tight_layout(
    pad=0.6
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    PNG_FILE,
    dpi=600,
    bbox_inches="tight",
)

plt.savefig(
    PDF_FILE,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# DONE
# ============================================================

print("\nSaved:")
print(PNG_FILE)
print(PDF_FILE)

In [ ]:
# ======================================================================
# FIGURE 5 — FINAL UPDATED MANUSCRIPT VERSION
#
# A = Internal embedding-level ablation
# B = KELM embedding-level ablation
# C = Internal sequence-level alanine-scanning faithfulness
# D = KELM sequence-level alanine-scanning faithfulness
#
# A/B:
#   Consensus-hotspot embedding ablation vs matched random ablation
#
# C/D:
#   Mean sequence-level faithfulness effect, F
#   Error bars = bootstrap 95% CI
#
# Style:
#   - compact 2 × 2
#   - all text bold
#   - lighter raw points in A/B
#   - reduced x-label rotation
#   - common scales within A/B and C/D
#   - 600 dpi PNG + PDF + SVG
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import gridspec
from matplotlib.patches import Patch


# ======================================================================
# PATHS
# ======================================================================

PROJECT = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

FAITH_DIR = (
    PROJECT
    / "06_xai"
    / "faithfulness"
)

RESULTS = (
    PROJECT
    / "07_results"
)

OUT_DIR = (
    RESULTS
    / "figures_main"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MUTATION_SUMMARY_FILE = (
    RESULTS
    / "tables_SI"
    / "sequence_mutation_faithfulness"
    / "sequence_level_mutation_faithfulness_summary.csv"
)


# ======================================================================
# MODELS / DATASETS
# ======================================================================

MODELS = [
    "ESM2_320",
    "ESM2_640",
    "ESM2_1280",
    "ProtT5",
]

MODEL_LABELS = [
    "ESM2-320",
    "ESM2-640",
    "ESM2-1280",
    "ProtT5",
]

DATASETS = [
    "internal_test",
    "kelm_external",
]


# ======================================================================
# VERIFY FILE
# ======================================================================

if not MUTATION_SUMMARY_FILE.exists():

    raise FileNotFoundError(
        f"Mutation summary not found:\n"
        f"{MUTATION_SUMMARY_FILE}"
    )

print(
    "✓ Mutation summary:",
    MUTATION_SUMMARY_FILE
)


# ======================================================================
# STYLE
# ======================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        10.5,

    "font.weight":
        "bold",

    "axes.labelsize":
        11.2,

    "axes.labelweight":
        "bold",

    "xtick.labelsize":
        9.2,

    "ytick.labelsize":
        9.2,

    "legend.fontsize":
        8.2,

    "axes.linewidth":
        1.1,

    "xtick.major.width":
        1.0,

    "ytick.major.width":
        1.0,

    "xtick.major.size":
        4,

    "ytick.major.size":
        4,

    "savefig.dpi":
        600,
})


# ======================================================================
# HELPERS
# ======================================================================

def find_col(
    df,
    candidates,
    contains=None,
):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    if contains:

        for c in df.columns:

            lc = c.lower()

            if all(
                x.lower() in lc
                for x in contains
            ):

                return c

    return None


def bold_ticks(
    ax
):

    for tick in ax.get_xticklabels():

        tick.set_fontweight(
            "bold"
        )

    for tick in ax.get_yticklabels():

        tick.set_fontweight(
            "bold"
        )


def add_panel_letter(
    ax,
    letter
):

    ax.text(
        -0.105,
        1.035,

        letter,

        transform=ax.transAxes,

        fontsize=18,
        fontweight="bold",

        ha="right",
        va="top",

        clip_on=False,
        zorder=100,
    )


# ======================================================================
# LOAD EXISTING EMBEDDING-ABLATION DATA
# ======================================================================

def read_faithfulness(
    model,
    dataset
):

    file = (
        FAITH_DIR
        / model
        / dataset
        / "per_sequence_faithfulness.csv"
    )

    if not file.exists():

        raise FileNotFoundError(
            f"Missing faithfulness file:\n"
            f"{file}"
        )

    df = pd.read_csv(
        file
    )

    df[
        "model_plot"
    ] = model

    df[
        "dataset_plot"
    ] = dataset

    return df


frames = []

for model in MODELS:

    for dataset in DATASETS:

        frames.append(
            read_faithfulness(
                model,
                dataset
            )
        )


faith = pd.concat(
    frames,
    ignore_index=True
)


# ======================================================================
# DETECT EXISTING ABLATION COLUMNS
# ======================================================================

hotspot_drop_col = find_col(
    faith,

    [
        "hotspot_drop",
        "consensus_hotspot_drop",
        "hotspot_probability_drop",
        "consensus_drop",
        "hotspot_ablation_drop",
    ],

    contains=[
        "hotspot",
        "drop",
    ],
)


random_drop_col = find_col(
    faith,

    [
        "random_drop",
        "matched_random_drop",
        "random_probability_drop",
        "random_ablation_drop",
        "mean_random_drop",
    ],

    contains=[
        "random",
        "drop",
    ],
)


if hotspot_drop_col is None:

    raise KeyError(
        "Could not detect hotspot-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


if random_drop_col is None:

    raise KeyError(
        "Could not detect random-drop column.\n"
        f"Columns:\n{faith.columns.tolist()}"
    )


print(
    "\nEmbedding-level columns:"
)

print(
    "Hotspot:",
    hotspot_drop_col
)

print(
    "Random :",
    random_drop_col
)


# ======================================================================
# LOAD SEQUENCE-MUTATION SUMMARY
# ======================================================================

mutation = pd.read_csv(
    MUTATION_SUMMARY_FILE
)


mutation[
    "model"
] = pd.Categorical(
    mutation[
        "model"
    ],

    categories=MODELS,

    ordered=True,
)


mutation = mutation.sort_values(
    [
        "dataset",
        "model",
    ]
)


# ======================================================================
# FIGURE CANVAS
# ======================================================================

fig = plt.figure(
    figsize=(12.0, 8.0)
)


gs = gridspec.GridSpec(
    2,
    2,

    figure=fig,

    left=0.105,
    right=0.985,

    bottom=0.095,
    top=0.965,

    wspace=0.24,
    hspace=0.31,
)


axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

axD = fig.add_subplot(
    gs[1, 1]
)


# ======================================================================
# PANELS A/B — EMBEDDING ABLATION
# ======================================================================

def perturbation_panel(
    ax,
    dataset,
):

    positions = np.arange(
        len(MODELS)
    )

    offset = 0.15

    box_width = 0.24


    for i, model in enumerate(
        MODELS
    ):

        sub = faith[
            (
                faith[
                    "model_plot"
                ]
                == model
            )
            &
            (
                faith[
                    "dataset_plot"
                ]
                == dataset
            )
        ]


        hotspot = (
            sub[
                hotspot_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        random_control = (
            sub[
                random_drop_col
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        # --------------------------------------------------------------
        # HOTSPOT ABLATION BOXPLOT
        # --------------------------------------------------------------

        ax.boxplot(
            hotspot,

            positions=[
                i - offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="0.82",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # RANDOM ABLATION BOXPLOT
        # --------------------------------------------------------------

        ax.boxplot(
            random_control,

            positions=[
                i + offset
            ],

            widths=box_width,

            patch_artist=True,

            showfliers=False,

            medianprops=dict(
                linewidth=1.5,
            ),

            boxprops=dict(
                linewidth=0.9,
                facecolor="white",
                edgecolor="black",
            ),

            whiskerprops=dict(
                linewidth=0.9,
            ),

            capprops=dict(
                linewidth=0.9,
            ),
        )


        # --------------------------------------------------------------
        # LIGHT RAW POINTS
        # --------------------------------------------------------------

        rng = np.random.default_rng(
            100 + i
        )


        if len(
            hotspot
        ):

            jitter = rng.normal(
                i - offset,
                0.020,
                len(hotspot),
            )

            ax.scatter(
                jitter,
                hotspot,

                s=5,
                alpha=0.075,

                linewidths=0,

                zorder=1,
            )


        if len(
            random_control
        ):

            jitter = rng.normal(
                i + offset,
                0.020,
                len(random_control),
            )

            ax.scatter(
                jitter,
                random_control,

                s=5,
                alpha=0.075,

                linewidths=0,

                zorder=1,
            )


    # --------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",
        linewidth=1.0,

        alpha=0.75,
    )


    # --------------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------------

    ax.set_xticks(
        positions
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=13,
        ha="right",
    )


    # --------------------------------------------------------------
    # Y AXIS
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Prediction change, Δp",

        fontsize=11.2,
        fontweight="bold",

        labelpad=7,
    )


    bold_ticks(
        ax
    )


    # --------------------------------------------------------------
    # LEGEND
    # --------------------------------------------------------------

    legend_handles = [

        Patch(
            facecolor="0.82",
            edgecolor="black",
            label="Consensus-hotspot ablation",
        ),

        Patch(
            facecolor="white",
            edgecolor="black",
            label="Matched random ablation",
        ),
    ]


    leg = ax.legend(
        handles=legend_handles,

        frameon=False,

        loc="upper right",

        bbox_to_anchor=(
            0.99,
            0.995,
        ),

        fontsize=8.0,

        handlelength=1.5,

        borderaxespad=0.1,
    )


    for text in leg.get_texts():

        text.set_fontweight(
            "bold"
        )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL A
# ======================================================================

perturbation_panel(
    axA,
    "internal_test",
)


# ======================================================================
# PANEL B
# ======================================================================

perturbation_panel(
    axB,
    "kelm_external",
)


# ======================================================================
# COMMON Y RANGE FOR A/B
# ======================================================================

combined_values = faith[
    [
        hotspot_drop_col,
        random_drop_col,
    ]
].astype(
    float
).values.flatten()


combined_values = (
    combined_values[
        np.isfinite(
            combined_values
        )
    ]
)


if len(
    combined_values
):

    q_low = np.quantile(
        combined_values,
        0.005,
    )

    q_high = np.quantile(
        combined_values,
        0.995,
    )

    span = (
        q_high
        - q_low
    )


    ymin = min(
        -0.10,
        q_low
        - 0.10 * span,
    )

    ymax = max(
        0.20,
        q_high
        + 0.10 * span,
    )


    axA.set_ylim(
        ymin,
        ymax,
    )

    axB.set_ylim(
        ymin,
        ymax,
    )


# ======================================================================
# PANELS C/D — SEQUENCE-LEVEL FAITHFULNESS
# ======================================================================

def mutation_faithfulness_panel(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    x = np.arange(
        len(MODELS)
    )


    means = (
        sub[
            "mean_faithfulness_effect"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_low = (
        sub[
            "faithfulness_CI_low"
        ]
        .astype(float)
        .to_numpy()
    )


    ci_high = (
        sub[
            "faithfulness_CI_high"
        ]
        .astype(float)
        .to_numpy()
    )


    lower_error = (
        means
        - ci_low
    )

    upper_error = (
        ci_high
        - means
    )


    yerr = np.vstack(
        [
            lower_error,
            upper_error,
        ]
    )


    # --------------------------------------------------------------
    # POINT + 95% CI
    # --------------------------------------------------------------

    ax.errorbar(
        x,
        means,

        yerr=yerr,

        fmt="o",

        markersize=7,

        markeredgecolor="black",
        markeredgewidth=0.9,

        capsize=4,
        capthick=1.2,

        elinewidth=1.3,

        linewidth=0,

        zorder=3,
    )


    # --------------------------------------------------------------
    # ZERO REFERENCE
    # --------------------------------------------------------------

    ax.axhline(
        0,

        linestyle="--",

        linewidth=1.0,

        alpha=0.75,

        zorder=1,
    )


    # --------------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------------

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        MODEL_LABELS,

        rotation=13,
        ha="right",
    )


    # --------------------------------------------------------------
    # Y AXIS
    # --------------------------------------------------------------

    ax.set_ylabel(
        "Sequence-level faithfulness, F",

        fontsize=11.2,

        fontweight="bold",

        labelpad=7,
    )


    bold_ticks(
        ax
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ======================================================================
# PANEL C
# ======================================================================

mutation_faithfulness_panel(
    axC,
    "internal_test",
)


# ======================================================================
# PANEL D
# ======================================================================

mutation_faithfulness_panel(
    axD,
    "kelm_external",
)


# ======================================================================
# COMMON Y RANGE FOR C/D
# ======================================================================

mutation_ci_min = (
    mutation[
        "faithfulness_CI_low"
    ]
    .astype(float)
    .min()
)

mutation_ci_max = (
    mutation[
        "faithfulness_CI_high"
    ]
    .astype(float)
    .max()
)


mutation_span = (
    mutation_ci_max
    - min(
        0,
        mutation_ci_min
    )
)


mutation_ymin = min(
    -0.025,
    mutation_ci_min
    - 0.08 * mutation_span,
)

mutation_ymax = (
    mutation_ci_max
    + 0.12 * mutation_span
)


axC.set_ylim(
    mutation_ymin,
    mutation_ymax,
)

axD.set_ylim(
    mutation_ymin,
    mutation_ymax,
)


# ======================================================================
# SIGNIFICANCE STARS FROM ACTUAL WILCOXON P VALUES
# ======================================================================

def p_to_star(
    p
):

    if p < 0.001:
        return "***"

    elif p < 0.01:
        return "**"

    elif p < 0.05:
        return "*"

    return ""


def add_mutation_significance(
    ax,
    dataset,
):

    sub = (
        mutation[
            mutation[
                "dataset"
            ]
            == dataset
        ]
        .copy()
        .set_index(
            "model"
        )
        .loc[
            MODELS
        ]
    )


    for i, model in enumerate(
        MODELS
    ):

        p = float(
            sub.loc[
                model,
                "wilcoxon_p",
            ]
        )

        upper = float(
            sub.loc[
                model,
                "faithfulness_CI_high",
            ]
        )

        star = p_to_star(
            p
        )


        if star:

            ax.text(
                i,
                upper
                + 0.015,

                star,

                ha="center",
                va="bottom",

                fontsize=10.5,

                fontweight="bold",

                clip_on=False,
            )


add_mutation_significance(
    axC,
    "internal_test",
)

add_mutation_significance(
    axD,
    "kelm_external",
)


# ======================================================================
# PANEL LETTERS
# ======================================================================

add_panel_letter(
    axA,
    "A",
)

add_panel_letter(
    axB,
    "B",
)

add_panel_letter(
    axC,
    "C",
)

add_panel_letter(
    axD,
    "D",
)


# ======================================================================
# FINAL FORMATTING
# ======================================================================

for ax in [
    axA,
    axB,
    axC,
    axD,
]:

    ax.tick_params(
        axis="both",

        width=1.0,

        length=4,
    )

    bold_ticks(
        ax
    )


# ======================================================================
# SAVE
# ======================================================================

FIG_BASE = (
    OUT_DIR
    / "Figure5_Final_Dual_Faithfulness"
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".png"
    ),

    dpi=600,

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".pdf"
    ),

    bbox_inches="tight",
)


fig.savefig(
    FIG_BASE.with_suffix(
        ".svg"
    ),

    bbox_inches="tight",
)


plt.show()


print(
    "\nSaved:"
)

print(
    FIG_BASE.with_suffix(
        ".png"
    )
)

print(
    FIG_BASE.with_suffix(
        ".pdf"
    )
)

print(
    FIG_BASE.with_suffix(
        ".svg"
    )
)

In [ ]:
# ============================================================
# FIGURE Sx
# Robustness of residue enrichment across hotspot thresholds
#
# A = Internal test
# B = KELM external
#
# Heatmap value:
#   log2(Odds Ratio)
#
# Significance:
#   *  FDR < 0.05
#
# Rows:
#   K, R, L, Q, G, M, Y, F, N
#
# Columns:
#   10%, 15%, 20%, 25%, 30%
#
# Saves:
#   PNG 600 dpi
#   PDF
#   SVG
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/pLM4CPP_XAI_2026"
)

INPUT_FILE = (
    PROJECT_DIR
    / "07_results"
    / "tables_SI"
    / "hotspot_threshold_robustness_all_residues.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "07_results"
    / "figures_SI"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_BASE = (
    OUTPUT_DIR
    / "FigureS_Threshold_Robustness_Heatmap"
)

assert INPUT_FILE.exists(), (
    f"Input file not found:\n{INPUT_FILE}"
)

print("Input:", INPUT_FILE)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# SETTINGS
# ============================================================

RESIDUE_ORDER = [
    "K",
    "R",
    "L",
    "Q",
    "G",
    "M",
    "Y",
    "F",
    "N",
]

THRESHOLD_ORDER = [
    10,
    15,
    20,
    25,
    30,
]

DATASETS = [
    "internal_test",
    "kelm_external",
]

DATASET_LABELS = {
    "internal_test": "Internal test",
    "kelm_external": "KELM external",
}


# ============================================================
# CALCULATE log2(OR)
# ============================================================

df = df.copy()

df["log2_OR"] = np.log2(
    df["odds_ratio"].astype(float)
)


# ============================================================
# BUILD MATRICES
# ============================================================

def build_matrix(
    dataset_name,
    value_col,
):

    sub = df[
        df["dataset"] == dataset_name
    ].copy()

    matrix = (
        sub
        .pivot(
            index="residue",
            columns="threshold_pct",
            values=value_col,
        )
        .reindex(
            index=RESIDUE_ORDER,
            columns=THRESHOLD_ORDER,
        )
    )

    return matrix


internal_log2 = build_matrix(
    "internal_test",
    "log2_OR",
)

kelm_log2 = build_matrix(
    "kelm_external",
    "log2_OR",
)

internal_fdr = build_matrix(
    "internal_test",
    "fdr",
)

kelm_fdr = build_matrix(
    "kelm_external",
    "fdr",
)


print("\nInternal log2(OR):")
display(internal_log2)

print("\nKELM log2(OR):")
display(kelm_log2)


# ============================================================
# SHARED COLOR SCALE
# ============================================================

all_values = np.concatenate(
    [
        internal_log2.to_numpy().flatten(),
        kelm_log2.to_numpy().flatten(),
    ]
)

all_values = all_values[
    np.isfinite(all_values)
]

max_abs = np.max(
    np.abs(all_values)
)

# symmetric range around zero
vmin = -max_abs
vmax = max_abs

print(
    "\nShared color scale:",
    round(vmin, 2),
    "to",
    round(vmax, 2),
)


# ============================================================
# STYLE
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
})


# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(8.2, 4.2),
    constrained_layout=True,
)


# ============================================================
# HELPER TO DRAW ONE HEATMAP
# ============================================================

def draw_heatmap(
    ax,
    value_matrix,
    fdr_matrix,
    dataset_label,
    panel_letter,
):

    im = ax.imshow(
        value_matrix.to_numpy(),
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap="coolwarm",
        interpolation="nearest",
    )

    # --------------------------------------------------------
    # X ticks
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(
            len(THRESHOLD_ORDER)
        )
    )

    ax.set_xticklabels(
        [
            f"{x}%"
            for x in THRESHOLD_ORDER
        ],
        fontweight="bold",
    )

    # --------------------------------------------------------
    # Y ticks
    # --------------------------------------------------------

    ax.set_yticks(
        np.arange(
            len(RESIDUE_ORDER)
        )
    )

    ax.set_yticklabels(
        RESIDUE_ORDER,
        fontweight="bold",
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Hotspot threshold",
        fontweight="bold",
    )

    ax.set_title(
        dataset_label,
        fontweight="bold",
        pad=8,
    )

    # --------------------------------------------------------
    # Cell annotations
    # --------------------------------------------------------

    values = value_matrix.to_numpy()
    fdrs = fdr_matrix.to_numpy()

    for i in range(
        values.shape[0]
    ):

        for j in range(
            values.shape[1]
        ):

            value = values[i, j]
            fdr = fdrs[i, j]

            if not np.isfinite(value):
                continue

            # text contrast
            normalized = abs(value) / max_abs

            text_color = (
                "white"
                if normalized > 0.55
                else "black"
            )

            sig = (
                "*"
                if fdr < 0.05
                else ""
            )

            ax.text(
                j,
                i,
                f"{value:.2f}{sig}",
                ha="center",
                va="center",
                fontsize=8.3,
                fontweight="bold",
                color=text_color,
            )

    # --------------------------------------------------------
    # Thin cell borders
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(
            -0.5,
            len(THRESHOLD_ORDER),
            1
        ),
        minor=True,
    )

    ax.set_yticks(
        np.arange(
            -0.5,
            len(RESIDUE_ORDER),
            1
        ),
        minor=True,
    )

    ax.grid(
        which="minor",
        linewidth=0.7,
        color="white",
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    # --------------------------------------------------------
    # Panel letter
    # --------------------------------------------------------

    ax.text(
        -0.16,
        1.06,
        panel_letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )

    # bold ticks
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    return im


# ============================================================
# PANEL A
# ============================================================

im = draw_heatmap(
    axes[0],
    internal_log2,
    internal_fdr,
    "Internal test",
    "A",
)


# ============================================================
# PANEL B
# ============================================================

draw_heatmap(
    axes[1],
    kelm_log2,
    kelm_fdr,
    "KELM external",
    "B",
)


# ============================================================
# SHARED Y LABEL
# ============================================================

axes[0].set_ylabel(
    "Residue",
    fontweight="bold",
)


# ============================================================
# COLORBAR
# ============================================================

cbar = fig.colorbar(
    im,
    ax=axes,
    shrink=0.90,
    pad=0.03,
)

cbar.set_label(
    "log₂(Odds ratio)",
    fontweight="bold",
)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight(
        "bold"
    )


# ============================================================
# SAVE
# ============================================================

fig.savefig(
    FIG_BASE.with_suffix(".png"),
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIG_BASE.with_suffix(".pdf"),
    bbox_inches="tight",
)

fig.savefig(
    FIG_BASE.with_suffix(".svg"),
    bbox_inches="tight",
)

plt.show()


print("\nSaved:")
print(FIG_BASE.with_suffix(".png"))
print(FIG_BASE.with_suffix(".pdf"))
print(FIG_BASE.with_suffix(".svg"))